# Sentiment experiment - TV3
TV2 completion and continuation authorized by user on 2026-10-28. Data approval status remains unchanged; local environment snapshot is not the leader lock.


In [1]:
from pathlib import Path
import json, sys, subprocess
ROOT = Path.cwd()
if not (ROOT / "ml").exists():
    ROOT = ROOT.parent
print("Data status:", json.loads((ROOT / "reports/data/manifest.json").read_text(encoding="utf-8"))["source"]["approval_status"])
M1_APPROVED = True  # User explicitly authorized using local TV2 output.


Data status: pending_leader_review


In [2]:
assert M1_APPROVED, "M1 confirmation required"
result = subprocess.run([sys.executable, "-m", "ml.train_sentiment", "--data-dir", "data/raw", "--output-dir", "models", "--report-dir", "reports/metrics/sentiment"], cwd=ROOT, check=True, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


Winner: linear_svm | val macro_f1=0.7530

D:\N.ThanhNhan\Project_AI\student-feedback-ai\.venv\sentiment\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\DELL\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'



In [3]:
result = json.loads((ROOT / "reports/metrics/sentiment/comparison.json").read_text(encoding="utf-8"))
import pandas as pd
rows = [{"split": split, "algorithm": algorithm, **{k: metric[k] for k in ("accuracy", "macro_precision", "macro_recall", "macro_f1")}} for split in ("validation", "test") for algorithm, metric in result[split].items()]
print(pd.DataFrame(rows).to_string(index=False))
print("Selected on validation:", result["selection"]["algorithm"])


     split           algorithm  accuracy  macro_precision  macro_recall  macro_f1
validation                  nb  0.873026         0.582215      0.611139  0.595527
validation logistic_regression  0.907770         0.877831      0.671929  0.688945
validation          linear_svm  0.914719         0.849978      0.721628  0.752966
      test                  nb  0.857233         0.572207      0.604441  0.586950
      test logistic_regression  0.889766         0.823527      0.646977  0.648432
      test          linear_svm  0.897031         0.804905      0.685335  0.705104
Selected on validation: linear_svm


## Error analysis
Use the frozen validation winner. See reports/metrics/sentiment/error_analysis.json for confusion pairs and example IDs. Source sentences are not published. Do not tune from final test scores.
